## 1. Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings("ignore")

## 2. Load Feature-Engineered Dataset

In [ ]:
feature_path = r"C:\Users\admin\Downloads\feature_engineered_health_dataset.csv"

df = pd.read_csv(feature_path)

print("Shape:", df.shape)
df.head()

## 3. Correlation with Target

In [ ]:
corr = df.corr(numeric_only=True)

corr_target = corr['Health_Risk_Category'].sort_values(ascending=False)

pd.set_option('display.max_rows', None)
print(corr_target)

## 4. Define Features and Target

In [ ]:
X = df.drop([
    'Person_ID',
    'Health_Risk_Category',
    '10_Year_Obesity_Risk_Percent',
    '10_Year_Diabetes_Risk_Percent',
    '10_Year_CVD_Risk_Percent'
], axis=1)

y = df['Health_Risk_Category']

print("Features:", X.shape)
print("Target:", y.shape)
print(X.columns)

## 5. Stratified Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

## 6. Data Type and Missing-Value Check

In [ ]:
X_train = X_train.apply(pd.to_numeric)

print("X missing:")
print(X.isna().sum().sum())

print("y missing:")
print(y.isna().sum())

print("X_train missing:")
print(X_train.isna().sum().sum())

print("y_train missing:")
print(y_train.isna().sum())

print(X_train.dtypes.value_counts())

## 7. Check Train-Test Overlap

In [ ]:
train_ids = set(X_train.index)
test_ids = set(X_test.index)

print("Train-test index overlap:", len(train_ids.intersection(test_ids)))

## 8. Handle Class Imbalance with SMOTE

In [ ]:
# Install once if imbalanced-learn is not already installed:
# !pip install imbalanced-learn

from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42,
    sampling_strategy='not majority'
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Original X_train:", X_train.shape)
print("SMOTE X_train:", X_train_smote.shape)
print("Original class distribution:")
print(y_train.value_counts())
print("SMOTE class distribution:")
print(y_train_smote.value_counts())

In [ ]:
print("SMOTE missing values:", X_train_smote.isna().sum().sum())

## 9. Feature Scaling — Logistic Regression

In [ ]:
scaler = StandardScaler()

X_train_scaled_smote = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = scaler.transform(X_train)

## 10. Model 1 — Logistic Regression

In [ ]:
lr = LogisticRegression()
lr.fit(X_train_scaled_smote, y_train_smote)

In [ ]:
y_pred_lr = lr.predict(X_test_scaled)
y_pred_lr_tr = lr.predict(X_train_scaled)

In [ ]:
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=['Low', 'Moderate', 'High']
))

accuracy_lr = accuracy_score(y_test, y_pred_lr)
accuracy_lr_tr = accuracy_score(y_train, y_pred_lr_tr)

precision_lr = precision_score(y_test, y_pred_lr, average='macro')
recall_lr = recall_score(y_test, y_pred_lr, average='macro')
recall_lr_tr = recall_score(y_train, y_pred_lr_tr, average='macro')

f1_lr = f1_score(y_test, y_pred_lr, average='macro')
f1_lr_tr = f1_score(y_train, y_pred_lr_tr, average='macro')

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr)).plot()
plt.show()

print("Test accuracy:", accuracy_lr)
print("Test macro F1:", f1_lr)

In [ ]:
high_risk_index = y_test == 2

print("Actual High Risk Cases:", high_risk_index.sum())
print(
    "Correctly Predicted High Risk:",
    ((y_test == 2) & (y_pred_lr == 2)).sum()
)

## 11. Model 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5
)

rf.fit(X_train_smote, y_train_smote)

In [ ]:
y_pred_rf = rf.predict(X_test)
y_pred_rf_tr = rf.predict(X_train)

In [ ]:
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=['Low', 'Moderate', 'High']
))

accuracy_rf = accuracy_score(y_test, y_pred_rf)
accuracy_rf_tr = accuracy_score(y_train, y_pred_rf_tr)

precision_rf = precision_score(y_test, y_pred_rf, average='macro')
recall_rf = recall_score(y_test, y_pred_rf, average='macro')
recall_rf_tr = recall_score(y_train, y_pred_rf_tr, average='macro')

f1_rf = f1_score(y_test, y_pred_rf, average='macro')
f1_rf_tr = f1_score(y_train, y_pred_rf_tr, average='macro')

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_rf)).plot()
plt.show()

print("Test accuracy:", accuracy_rf)
print("Test macro F1:", f1_rf)

### Random Forest Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators': [150, 200, 250],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 3, 4],
    'min_samples_leaf': [2, 3, 4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid.fit(X_train_smote, y_train_smote)

print("Best parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

In [ ]:
best_rf = grid.best_estimator_
best_rf.fit(X_train_smote, y_train_smote)

y_pred_best = best_rf.predict(X_test)

print(classification_report(
    y_test,
    y_pred_best,
    target_names=['Low', 'Moderate', 'High']
))

accuracy_rf_best = accuracy_score(y_test, y_pred_best)
precision_rf_best = precision_score(y_test, y_pred_best, average='macro')
recall_rf_best = recall_score(y_test, y_pred_best, average='macro')
f1_rf_best = f1_score(y_test, y_pred_best, average='macro')

In [ ]:
high_risk_index = y_test == 2

print("Actual High Risk Cases:", high_risk_index.sum())
print(
    "Correctly Predicted High Risk:",
    ((y_test == 2) & (y_pred_rf == 2)).sum()
)
print(
    "Correctly Predicted High Risk after tuning:",
    ((y_test == 2) & (y_pred_best == 2)).sum()
)

## 12. Model 3 — XGBoost

In [ ]:
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    eval_metric='mlogloss'
)

xgb.fit(X_train_smote, y_train_smote)

In [ ]:
y_pred_xgb = xgb.predict(X_test)
y_pred_xgb_tr = xgb.predict(X_train)

print(classification_report(
    y_test,
    y_pred_xgb,
    target_names=['Low', 'Moderate', 'High']
))

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
accuracy_xgb_tr = accuracy_score(y_train, y_pred_xgb_tr)

precision_xgb = precision_score(y_test, y_pred_xgb, average='macro')
recall_xgb = recall_score(y_test, y_pred_xgb, average='macro')
recall_xgb_tr = recall_score(y_train, y_pred_xgb_tr, average='macro')

f1_xgb = f1_score(y_test, y_pred_xgb, average='macro')
f1_xgb_tr = f1_score(y_train, y_pred_xgb_tr, average='macro')

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_xgb)).plot()
plt.show()

print("Test accuracy:", accuracy_xgb)
print("Test macro F1:", f1_xgb)

## 13. Cross Validation

In [ ]:
cv_scores = cross_val_score(
    lr,
    X_train,
    y_train,
    cv=5,
    scoring='f1_macro'
)

print("Logistic Regression CV Scores:", cv_scores)
print("Mean F1:", cv_scores.mean())

In [ ]:
cv_scores = cross_val_score(
    xgb,
    X_train,
    y_train,
    cv=5,
    scoring='f1_macro'
)

print("XGBoost CV Scores:", cv_scores)
print("Mean F1:", cv_scores.mean())

In [ ]:
high_risk_index = y_test == 2

print("Actual High Risk Cases:", high_risk_index.sum())
print(
    "Correctly Predicted High Risk:",
    ((y_test == 2) & (y_pred_xgb == 2)).sum()
)

## 14. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [accuracy_lr, accuracy_rf_best, accuracy_xgb],
    'Precision': [precision_lr, precision_rf_best, precision_xgb],
    'Recall': [recall_lr, recall_rf_best, recall_xgb],
    'F1 Score': [f1_lr, f1_rf_best, f1_xgb]
})

results.sort_values(by='F1 Score', ascending=False)

In [ ]:
print("LOGISTIC REGRESSION")
print(classification_report(
    y_test, y_pred_lr,
    target_names=['Low', 'Moderate', 'High']
))

print("BEST RANDOM FOREST")
print(classification_report(
    y_test, y_pred_best,
    target_names=['Low', 'Moderate', 'High']
))

print("XGBOOST")
print(classification_report(
    y_test, y_pred_xgb,
    target_names=['Low', 'Moderate', 'High']
))

## 15. Export Model Results

In [ ]:
results_path = r"C:\Users\admin\Downloads\Model_comparison.csv"
results.to_csv(results_path, index=False)

cm_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred_lr),
    index=['Low', 'Moderate', 'High'],
    columns=['Low', 'Moderate', 'High']
)

cm_path = r"C:\Users\admin\Downloads\confusion_matrix.csv"
cm_df.to_csv(cm_path)

pred_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred_lr
})

pred_path = r"C:\Users\admin\Downloads\predictions.csv"
pred_df.to_csv(pred_path, index=False)

importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': abs(lr.coef_).mean(axis=0)
})

importance.sort_values(
    by='Coefficient',
    ascending=False,
    inplace=True
)

importance_path = r"C:\Users\admin\Downloads\feature_importance.csv"
importance.to_csv(importance_path, index=False)

print("Exported:")
print(results_path)
print(cm_path)
print(pred_path)
print(importance_path)